# Reproducibilidad y bibliografía

Este capítulo documenta lo necesario para reproducir los resultados: el entorno de software,
la semilla, la tabla maestra de experimentos y el orden de ejecución.

## Entorno

La tabla lista las versiones de las bibliotecas principales con que se obtuvieron los resultados.

In [1]:
# Versiones de Python y de las bibliotecas principales
import sys, platform, json
sys.path.insert(0, "..")
from importlib.metadata import version
import pandas as pd
from src.config import SEED, RESULTADOS
libs = ["numpy", "pandas", "scipy", "scikit-learn", "imbalanced-learn", "xgboost", "optuna", "deap", "shap", "lime",
        "statsmodels", "arch", "ngboost", "faiss-cpu", "dash", "dash-bootstrap-components", "plotly", "jupyter-book"]
pd.DataFrame({"versión": {l: version(l) for l in libs} | {"python": platform.python_version()}})

,versión
numpy,1.26.4
pandas,2.2.3
scipy,1.14.1
scikit-learn,1.5.2
imbalanced-learn,0.12.4
xgboost,2.1.3
optuna,4.1.0
deap,1.4.1
shap,0.46.0
lime,0.2.0.1


Todas las versiones están fijadas en `requirements-dev.txt` (experimentación y libro) y en
`requirements.txt` (aplicación interactiva en producción). Los resultados se obtuvieron en Windows 11 con un
procesador Intel Core i7-1255U y 15,7 GB de memoria, sin GPU.

## Semillas

La semilla global es `SEED = 42`, definida una sola vez en `src/config.py`. La función
`fijar_semillas` la propaga a los generadores de Python y numpy; cada estimador la recibe en
`random_state` (scikit-learn, XGBoost, imbalanced-learn y el modelo propuesto), el muestreador
TPE de Optuna en `seed` y el algoritmo genético la usa tanto en el generador de numpy que crea
la población inicial como en el módulo `random` que emplea DEAP en la selección, el cruce y la
mutación. La sensibilidad a la semilla se evaluó explícitamente con cinco semillas ({doc}`03a_clasificacion` y
{doc}`06_validacion`).

## Tabla maestra de experimentos

La celda siguiente reúne los resultados de todas las optimizaciones en una tabla maestra, la
guarda en disco y muestra sus primeras filas.

In [2]:
# Tabla maestra: una fila por combinación con media y desviación de cada métrica, tiempo y semilla
from src.analysis import tabla_pliegues
partes = []
for carpeta, tarea in (("clf_calamar", "clasificación"), ("reg_calamar", "regresión"), ("original_calamar", "clasificación")):
    t = tabla_pliegues(carpeta); t["tarea"] = tarea
    partes.append(t)
largo = pd.concat(partes, ignore_index=True)
from src.analysis import cargar_carpeta
params = {(r["modelo"], r["balanceo"], r["optimizador"], r["pliegue"]): json.dumps(r["params"])
          for c in ("clf_calamar", "reg_calamar", "original_calamar") for r in cargar_carpeta(c)}
largo["hiperparametros"] = [params[(a, b, c, d)] for a, b, c, d in zip(largo.modelo, largo.balanceo, largo.optimizador, largo.pliegue)]
largo["semilla"] = SEED
metricas = [c for c in ("pr_auc", "roc_auc", "f1", "precision", "recall", "accuracy", "brier", "rmse", "mae", "r2", "mape") if c in largo]
agg = largo.groupby(["tarea", "modelo", "balanceo", "optimizador"]).agg(
    **{f"{m}_media": (m, "mean") for m in metricas}, **{f"{m}_desv": (m, "std") for m in metricas},
    tiempo_total_s=("t_opt", "sum"), evaluaciones=("n_eval", "sum"), pliegues=("pliegue", "nunique")).reset_index()
agg["semilla"] = SEED
largo.to_csv(RESULTADOS / "tabla_maestra_por_pliegue.csv", index=False)
agg.to_csv(RESULTADOS / "tabla_maestra.csv", index=False)
print(f"{len(agg)} combinaciones ({(agg.tarea == 'clasificación').sum()} de clasificación, "
      f"{(agg.tarea == 'regresión').sum()} de regresión) y {len(largo):,} registros por pliegue.")
agg.head(10)

147 combinaciones (119 de clasificación, 28 de regresión) y 1,764 registros por pliegue.


,tarea,modelo,balanceo,optimizador,pr_auc_media,roc_auc_media,f1_media,precision_media,recall_media,accuracy_media,...,accuracy_desv,brier_desv,rmse_desv,mae_desv,r2_desv,mape_desv,tiempo_total_s,evaluaciones,pliegues,semilla
0,clasificación,arbol,adasyn,bayes,0.823377,0.949944,0.747056,0.645116,0.929716,0.865033,...,0.056905,0.040648,NaN,NaN,NaN,NaN,603.568744,192,12,42
1,clasificación,arbol,adasyn,genetico,0.819597,0.945409,0.736796,0.637943,0.917673,0.860405,...,0.058249,0.041865,NaN,NaN,NaN,NaN,591.355474,192,12,42
2,clasificación,arbol,adasyn,grid,0.817331,0.946920,0.760622,0.660727,0.939898,0.871537,...,0.061336,0.046960,NaN,NaN,NaN,NaN,579.468026,192,12,42
3,clasificación,arbol,adasyn,random,0.811542,0.946119,0.755920,0.651948,0.939623,0.869594,...,0.061602,0.046401,NaN,NaN,NaN,NaN,640.133744,192,12,42
4,clasificación,arbol,balanced,bayes,0.834009,0.955161,0.774157,0.689797,0.911165,0.885921,...,0.047243,0.028558,NaN,NaN,NaN,NaN,209.150589,192,12,42
5,clasificación,arbol,balanced,genetico,0.858403,0.958021,0.779494,0.691585,0.920481,0.888873,...,0.040021,0.025360,NaN,NaN,NaN,NaN,209.610429,192,12,42
6,clasificación,arbol,balanced,grid,0.834966,0.953047,0.765905,0.668263,0.930796,0.872748,...,0.060749,0.031675,NaN,NaN,NaN,NaN,190.795915,192,12,42
7,clasificación,arbol,balanced,random,0.833961,0.949313,0.769216,0.687024,0.909654,0.883668,...,0.052558,0.028839,NaN,NaN,NaN,NaN,214.925280,192,12,42
8,clasificación,arbol,ninguno,bayes,0.848508,0.954955,0.753114,0.731742,0.799189,0.888580,...,0.050047,0.026237,NaN,NaN,NaN,NaN,234.419083,192,12,42
9,clasificación,arbol,ninguno,genetico,0.842621,0.960224,0.785632,0.782409,0.799907,0.910824,...,0.027868,0.018810,NaN,NaN,NaN,NaN,224.630224,192,12,42


La tabla maestra (`results/tabla_maestra.csv`) contiene una fila por combinación con la media y
la desviación de cada métrica del bucle externo, el tiempo total, el número de evaluaciones y la
semilla; `results/tabla_maestra_por_pliegue.csv` agrega los hiperparámetros elegidos en cada
pliegue externo. Las 112 combinaciones de clasificación, las 28 de regresión y las variantes del
modelo propuesto, de los métodos distribucionales y de DLinear comparten el mismo esquema.

## Cómo reproducir los resultados

El README del repositorio enumera los comandos en orden. Los cómputos costosos (la validación
anidada de los 140 modelos, del modelo propuesto y de los métodos recientes) se guardan como resultados versionados en
`results/`, y los cuadernos de este libro los leen y regeneran tablas y figuras; el código que
los produce está en `scripts/` y `src/` y se ejecuta de principio a fin con los comandos del
README. Las pruebas unitarias se ejecutan con `python -m pytest`.

## Bibliografía

```{bibliography}
:style: unsrt
```